<a href="https://colab.research.google.com/github/suegy/tig-330/blob/main/notebooks/Flux-schnell-lowmem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y -q gradio gradio-client

# Install the known FLUX NF4 stack
!pip install -q \
    bitsandbytes==0.48.1 \
    diffusers==0.35.1 \
    peft==0.17.1 \
    protobuf==5.29.5 \
    sentencepiece==0.2.1 \
    transformers==4.56.1 \
    huggingface-hub==0.36.2 \
    accelerate

In [ ]:
import gc
import torch

from diffusers import FluxPipeline, FluxTransformer2DModel
from transformers import T5EncoderModel

MODEL = "aniketppanchal/flux.1-schnell-nf4-pkg"


def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
def generate_flux(
    prompt,
    width=768,
    height=768,
    steps=4,
    seed=42,
):
    # ============================================================
    # STAGE 1 — Prompt encoding
    # ============================================================

    text_encoder_2 = T5EncoderModel.from_pretrained(
        MODEL,
        subfolder="text_encoder_2",
        torch_dtype=torch.float16,
        device_map="cuda",
    )

    pipeline = FluxPipeline.from_pretrained(
        MODEL,
        text_encoder_2=text_encoder_2,
        transformer=None,
        vae=None,
        torch_dtype=torch.float16,
        device_map="cuda",
    )

    with torch.no_grad():
        prompt_embeds, pooled_prompt_embeds, _ = pipeline.encode_prompt(
            prompt=prompt,
            max_sequence_length=256,
        )

    del text_encoder_2
    del pipeline
    clear_vram()

    # ============================================================
    # STAGE 2 — FLUX transformer
    # ============================================================

    transformer = FluxTransformer2DModel.from_pretrained(
        MODEL,
        subfolder="transformer",
        torch_dtype=torch.float16,
        device_map="cuda",
    )

    pipeline = FluxPipeline.from_pretrained(
        MODEL,
        text_encoder=None,
        text_encoder_2=None,
        tokenizer=None,
        tokenizer_2=None,
        transformer=transformer,
        vae=None,
        dtype=torch.float16,
        device_map="cuda",
    )

    generator = torch.Generator("cuda").manual_seed(seed)

    with torch.no_grad():
        packed_latents = pipeline(
            height=height,
            width=width,
            num_inference_steps=steps,
            guidance_scale=0.0,
            prompt_embeds=prompt_embeds,
            pooled_prompt_embeds=pooled_prompt_embeds,
            output_type="latent",
            max_sequence_length=256,
            generator=generator,
        ).images

    del prompt_embeds
    del pooled_prompt_embeds
    del transformer
    del pipeline
    clear_vram()

    # ============================================================
    # STAGE 3 — VAE decoding
    # ============================================================

    pipeline = FluxPipeline.from_pretrained(
        MODEL,
        text_encoder=None,
        text_encoder_2=None,
        tokenizer=None,
        tokenizer_2=None,
        transformer=None,
        torch_dtype=torch.float16,
        device_map="cuda",
    )

    unpacked_latents = (
        pipeline._unpack_latents(
            packed_latents,
            height=height,
            width=width,
            vae_scale_factor=pipeline.vae_scale_factor,
        )
        / pipeline.vae.config.scaling_factor
        + pipeline.vae.config.shift_factor
    )

    # IMPORTANT:
    # Match the latent dtype to the VAE's actual parameter dtype.
    vae_param = next(pipeline.vae.parameters())

    unpacked_latents = unpacked_latents.to(
        device=vae_param.device,
        dtype=vae_param.dtype,
    )

    with torch.inference_mode():
        image_tensor = pipeline.vae.decode(
            unpacked_latents,
            return_dict=False,
        )[0]

    image = pipeline.image_processor.postprocess(
        image_tensor,
        output_type="pil",
    )[0]

    del packed_latents
    del unpacked_latents
    del image_tensor
    del pipeline
    clear_vram()

    return image

In [ ]:
prompt = """
An image for a website that is used for an academic seminar series in a colage style with a sign in the centre reading only "Positive Technology Visions" the text being spread over 3 lines.
In the background you have an artistic representation of science on the left and an artistic representation of the field of arts on the left united by technology in the centre,
creative, colourful, non-realtistic, impressionist, joyful
"""

image = generate_flux(
    prompt=prompt,
    width=1024,
    height=768,
    steps=7,
    seed=12345,
)

display(image)